# ⚛️ Quant Forge — Mathematical Derivations
## Quantum-Inspired NSE Financial Analytics Engine
### Rajnish Singh | Quant Forge Research

---
This notebook contains full derivations, proofs, and implementations
of all mathematical models in the Quant Forge platform.


## 1. Geometric Brownian Motion (GBM)

### Model Definition
The GBM SDE for asset price $S_t$:
$$dS_t = \mu S_t \, dt + \sigma S_t \, dW_t$$
where $W_t$ is a standard Brownian motion ($dW_t \sim \mathcal{N}(0, dt)$).

### Exact Solution via Itô's Lemma
Let $X_t = \ln S_t$. By Itô's lemma:
$$dX_t = \left(\mu - \frac{\sigma^2}{2}\right)dt + \sigma \, dW_t$$
This is a driftless SDE with constant coefficients. Integrating:
$$X_T - X_0 = \left(\mu - \frac{\sigma^2}{2}\right)T + \sigma W_T$$
Therefore:
$$\boxed{S_T = S_0 \exp\left[\left(\mu - \frac{\sigma^2}{2}\right)T + \sigma\sqrt{T}\, Z\right], \quad Z \sim \mathcal{N}(0,1)}$$

### Properties
- $\mathbb{E}[S_T] = S_0 e^{\mu T}$ (exponential growth at rate $\mu$)
- $\text{Var}(S_T) = S_0^2 e^{2\mu T}(e^{\sigma^2 T} - 1)$
- $\ln S_T \sim \mathcal{N}(\ln S_0 + (\mu - \sigma^2/2)T,\; \sigma^2 T)$

### MLE Calibration
From log-returns $r_i = \ln(S_i/S_{i-1})$:
$$\hat{\mu} = \bar{r} \cdot N + \frac{\hat{\sigma}^2}{2}, \qquad \hat{\sigma} = s_r \sqrt{N}$$
where $N = 252$ trading days.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.style as style
style.use('dark_background')

# GBM Simulation
np.random.seed(42)
S0, mu, sigma, T, dt = 1000, 0.12, 0.25, 1, 1/252
N = int(T/dt)
n_paths = 1000

# Exact simulation
Z = np.random.randn(n_paths, N)
log_increments = (mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z
paths = S0 * np.exp(np.hstack([np.zeros((n_paths,1)), np.cumsum(log_increments, axis=1)]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Paths
for i in range(100):
    axes[0].plot(paths[i], alpha=0.1, color='cyan', lw=0.5)
axes[0].plot(np.percentile(paths, 50, axis=0), color='gold', lw=2, label='Median')
axes[0].plot(np.percentile(paths, 5, axis=0), color='red', lw=2, ls='--', label='5th pct')
axes[0].plot(np.percentile(paths, 95, axis=0), color='green', lw=2, ls='--', label='95th pct')
axes[0].set_title(f'GBM Paths (μ={mu}, σ={sigma})', color='gold')
axes[0].legend()

# Terminal distribution
axes[1].hist(paths[:,-1], bins=50, color='cyan', alpha=0.7, density=True)
axes[1].axvline(np.mean(paths[:,-1]), color='gold', lw=2, label=f'Mean={np.mean(paths[:,-1]):.0f}')
axes[1].set_title('Terminal Price Distribution', color='gold')
axes[1].legend()
plt.tight_layout()
plt.savefig('gbm_simulation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Theoretical E[S_T] = {S0*np.exp(mu*T):.2f}')
print(f'Simulated  E[S_T] = {np.mean(paths[:,-1]):.2f}')

## 2. Value at Risk — Three Methods

### Definition
$$\text{VaR}_\alpha = \inf\{x \in \mathbb{R} : P(L > x) \leq 1 - \alpha\} = F_L^{-1}(\alpha)$$

### Method 1: Historical Simulation
Order statistics on empirical loss distribution $\{l_1, ..., l_n\}$:
$$\widehat{\text{VaR}}_\alpha = l_{(\lceil \alpha n \rceil)}$$

### Method 2: Parametric (Delta-Normal)
Assuming $r \sim \mathcal{N}(\mu_r, \sigma_r^2)$, losses $L = -r \sim \mathcal{N}(-\mu_r, \sigma_r^2)$:
$$\text{VaR}_\alpha = -\mu_r + z_\alpha \sigma_r, \quad z_\alpha = \Phi^{-1}(\alpha)$$
Expected Shortfall:
$$\text{ES}_\alpha = -\mu_r + \sigma_r \frac{\phi(z_\alpha)}{1-\alpha}$$
where $\phi$ is the standard normal PDF.

### Method 3: Cornish-Fisher Expansion
For non-normal returns with skewness $\gamma$ and excess kurtosis $\kappa$:
$$z_{CF} = z + \frac{z^2-1}{6}\gamma + \frac{z^3-3z}{24}\kappa - \frac{2z^3-5z}{36}\gamma^2$$
$$\text{VaR}_{\alpha}^{CF} = -(\mu_r + \sigma_r z_{CF})$$


## 3. GARCH(1,1) — Volatility Clustering

### Model
$$\varepsilon_t = \sigma_t z_t, \quad z_t \sim \mathcal{N}(0,1)$$
$$\sigma_t^2 = \omega + \alpha \varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

### Stationarity Condition
$\alpha + \beta < 1$ ensures covariance stationarity.

### Long-Run Variance
$$\bar{\sigma}^2 = \frac{\omega}{1 - \alpha - \beta}$$

### h-step Forecast
$$\sigma^2_{t+h|t} = \bar{\sigma}^2 + (\alpha+\beta)^{h-1}(\sigma^2_{t+1|t} - \bar{\sigma}^2)$$

### MLE
Log-likelihood (normal innovations):
$$\ell(\theta) = -\frac{T}{2}\ln(2\pi) - \frac{1}{2}\sum_{t=1}^T \left[\ln\sigma_t^2 + \frac{\varepsilon_t^2}{\sigma_t^2}\right]$$


## 4. Kalman Filter

### State-Space Model
State: $\mathbf{x}_t = [\text{level}, \text{trend}]^T$

**Transition**: $\mathbf{x}_{t+1} = F \mathbf{x}_t + \mathbf{w}_t$
$$F = \begin{pmatrix} 1 & 1 \\ 0 & 1 \end{pmatrix}, \quad \mathbf{w}_t \sim \mathcal{N}(\mathbf{0}, Q)$$

**Observation**: $y_t = H \mathbf{x}_t + v_t$
$$H = [1, 0], \quad v_t \sim \mathcal{N}(0, R)$$

### Kalman Recursion
**Predict**:
$$\mathbf{x}_{t|t-1} = F \mathbf{x}_{t-1|t-1}$$
$$P_{t|t-1} = F P_{t-1|t-1} F^T + Q$$

**Update**:
$$K_t = P_{t|t-1} H^T (H P_{t|t-1} H^T + R)^{-1}$$
$$\mathbf{x}_{t|t} = \mathbf{x}_{t|t-1} + K_t (y_t - H \mathbf{x}_{t|t-1})$$
$$P_{t|t} = (I - K_t H) P_{t|t-1}$$


## 5. Markowitz Portfolio Theory

### Efficient Frontier
**Primal problem** (minimize variance for target return $\mu^*$):
$$\min_{\mathbf{w}} \; \mathbf{w}^T \Sigma \mathbf{w} \quad \text{s.t.} \; \mathbf{w}^T \boldsymbol{\mu} = \mu^*, \; \mathbf{1}^T \mathbf{w} = 1, \; \mathbf{w} \geq \mathbf{0}$$

### Max Sharpe (Tangency Portfolio)
$$\max_{\mathbf{w}} \; \frac{\mathbf{w}^T \boldsymbol{\mu} - r_f}{\sqrt{\mathbf{w}^T \Sigma \mathbf{w}}}$$
Analytical solution (unconstrained): $\mathbf{w}^* \propto \Sigma^{-1}(\boldsymbol{\mu} - r_f \mathbf{1})$

### Risk Parity
Equal risk contribution: $RC_i = w_i (\Sigma \mathbf{w})_i = \sigma_p/n$ for all $i$
$$\text{Minimize: } \sum_{i=1}^n \left(RC_i - \frac{\sigma_p}{n}\right)^2$$


## 6. QAOA for Portfolio Optimization

### QUBO Formulation
Binary variable $x_i \in \{0,1\}$: asset $i$ selected.

**Objective**:
$$\min_{\mathbf{x}} \; \lambda_r \mathbf{x}^T \Sigma \mathbf{x} - \lambda_\mu \boldsymbol{\mu}^T \mathbf{x} + \lambda_c \left(\sum_i x_i - k\right)^2$$

### Ising Mapping
Substitution $x_i = (1 - z_i)/2$, $z_i \in \{\pm 1\}$:
$$H_{\text{Ising}} = \sum_i h_i z_i + \sum_{i<j} J_{ij} z_i z_j$$

### QAOA Ansatz
Variational quantum state of depth $p$:
$$|\psi(\boldsymbol{\beta}, \boldsymbol{\gamma})\rangle = \prod_{l=1}^p U_B(\beta_l) U_C(\gamma_l) |+\rangle^{\otimes n}$$
- $U_C(\gamma) = e^{-i\gamma H_C}$ (problem unitary: RZZ + RZ gates)
- $U_B(\beta) = e^{-i\beta H_B}$, $H_B = \sum_i X_i$ (mixing: RX gates)

### Classical Optimization
$$\boldsymbol{\beta}^*, \boldsymbol{\gamma}^* = \arg\min_{\boldsymbol{\beta},\boldsymbol{\gamma}} \langle \psi(\boldsymbol{\beta},\boldsymbol{\gamma}) | H_C | \psi(\boldsymbol{\beta},\boldsymbol{\gamma}) \rangle$$

Optimized via COBYLA (gradient-free, noise-robust).
